### Lab Assignment: Python Warmup and Logfile Analytics

### University of Virginia
### DS 5110: Big Data Systems
### Last Updated: January 23, 2026

---

This lab consists of two parts:

- Part 1 is the Python warmup
- Part 2 is the logfile analytics in PySpark

Answer the questions in this assignment, showing all code and solutions.

**Total points: 20**

---

### Part 1: Python Warmup

1) (1 PT) Rename this notebook to JupyterTutorial_[your_initials], where you will enter your initials in place of [your_initials].

2) (1 PT) In the cell below, enter a list of data science topics you find interesting.  Use the markdown style (you will need to change the style from the Code style).

#### I find the following areas interesting:
- Data Visualization, although we've not done much of that so far in this program, behind various plotting capabilities in R.
- Understanding the math and statistics "behind the scenes" that are the backbone of data science, although my command of these techniques is not as strong as I'd like.
  - *I really wish this program had an introductory course that covered key aspects of linear algebra, perhaps along with a primer in R and Python that connected all three together.*
- Learning the programming skills that enable working with these large data sets and drawing meaningful insights from them (this is probably my favorite part).

3) (1 PT) In the cell below, enter the following Python list:  

some_vals = [1, 6, 10, 44]  

You will use the Code style, run the cell, and print the list.

In [2]:
# Create the Python list
some_vals = [1, 6, 10, 44]
print(some_vals)

[1, 6, 10, 44]


4) (1 PT) Use a list comprehension to return a filtered list containing only the values greater than 6.  
Call this list *some_vals_filtered* and print it.

In [4]:
some_vals_filtered = [val for val in some_vals if val > 6]
print(some_vals_filtered)

[10, 44]


Next, a small pandas dataframe is constructed.

In [6]:
import pandas as pd

df = pd.DataFrame({'first_name': ['Andy','Crystal'],
                   'domain_facebook' : [1,1],
                   'domain_foursquare' : [0,0],
                   'age' : [20, 32]})
df

,first_name,domain_facebook,domain_foursquare,age
0,Andy,1,0,20
1,Crystal,1,0,32


5) (1 PT) In the cell below, write a list comprehension that returns the fields names in the dataframe `df` containing the string *domain*.  Run the cell to verify the correct result.

In [9]:
field_names = [col for col in df.columns if 'domain' in col]
print(field_names)

['domain_facebook', 'domain_foursquare']


6) (1 PT) Use the list comprehension from (5) to index into `df` and show the data for columns containing *domain*

In [11]:
# Is this what you mean?
df[field_names]

,domain_facebook,domain_foursquare
0,1,0
1,1,0


7) (1 PT) In the cell below, print the *domain_facebook* column

In [13]:
df['domain_facebook']

0    1
1    1
Name: domain_facebook, dtype: int64

8) (1 PT) In the cell below, print the row with index 1.

In [15]:
df[1:]

,first_name,domain_facebook,domain_foursquare,age
1,Crystal,1,0,32


9) (1 PT) Next, you will cube the *age* column of `df` and assign the result to a new column called *agecube*.

Specifically, call the `apply` method with a `lambda function` inside to cube the *age* column.  
Print the dataframe.

In [18]:
# Define the function.
cube_fx = lambda x: x**3

# Apply it to the df and create the new column.
df['agecube'] = df['age'].apply(cube_fx)

# Display the df.
df

,first_name,domain_facebook,domain_foursquare,age,agecube
0,Andy,1,0,20,8000
1,Crystal,1,0,32,32768


10) (1 PT) Given the list of strings below, form one string, placing semicolons between each word.  It should look like this:  

`'the;quick;brown;fox'`

Print the resulting string.

In [22]:
some_list = ['the','quick','brown','fox']

In [25]:
scrunched = ';'.join(str(word) for word in some_list)
print(scrunched)

the;quick;brown;fox


---

### Part 2: Logfile Analytics

Import modules for Spark Session and regex 

Note: regexes can be used to search strings for patterns. Here is a [reference](https://realpython.com/regex-python/?utm_source=chatgpt.com).

In [26]:
from pyspark.sql import SparkSession
import re

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/04 21:41:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


11) (1 PT) Read in the logfile.txt data

In [29]:
log_file = sc.textFile('logfile.txt')

12) (1 PT) Count the number of rows of data

In [31]:
log_file.count()

100000

13) (2 PTS) Show the first five lines containing WARN.  
    Write code to count and print the total number of lines containing WARN.

In [36]:
# Filter the log file data for lines that contain WARN.
log_file_WARN = log_file.filter(lambda line: "WARN" in line)

# Count the number of lines with WARN.
WARN_count = log_file_WARN.count()
print("WARN count: ", WARN_count)

# Show the first five lines that contain WARN.
log_file_WARN.take(5)

WARN count:  14823


['2026-01-01T08:00:09.764000 [WARN] scheduler: heartbeat ip=92.174.240.101 latency_ms=475 trace=nw6i1m40mwvc seq=19',
 '2026-01-01T08:00:12.828000 [WARN] api: partition reassigned ip=84.161.222.95 latency_ms=4691 trace=yht26vhtm0eh seq=26',
 '2026-01-01T08:00:14.718000 [WARN] auth: job completed ip=230.79.10.241 latency_ms=3732 trace=tedo8mtq5qu9 seq=29',
 '2026-01-01T08:00:20.259000 [WARN] api: shuffle completed ip=212.54.74.42 latency_ms=4063 trace=ej2u544h7nwc seq=40',
 '2026-01-01T08:00:28.523000 [WARN] auth: cache miss ip=242.94.66.209 latency_ms=753 trace=feqvh0k4dwun seq=57']

14) (2 PTS) Write a word count program to count the number of each of these log levels:  

- WARN
- INFO
- DEBUG
- ERROR

In [39]:
# Create a list of the log levels for using in the lambda function.
log_levels = ["WARN", "INFO", "DEBUG", "ERROR"]

# Use the power of Spark to reduce the log file text down to the log level key-value pairs.
log_counts = (
    log_file
        .flatMap(lambda line: [level for level in log_levels if level in line])
        .map(lambda level: (level, 1))
        .reduceByKey(lambda x, y: x + y)
)

# Collect the counts.
log_counts.collect()

[('INFO', 70083), ('ERROR', 9951), ('DEBUG', 5143), ('WARN', 14823)]

15. (2 PTS) Return the three log lines with the highest latency. This is reported in the log as `latency_ms`.

Note: There may be more than three lines tied for highest latency, in which case, just show three records.

In [46]:
# Use RegEx to help with this, since I see "re" was imported earlier.
# NOTE: I took help from MS Copilot to get the RegEx pattern...it's just an efficient use of an LLM.
# Parse out the latency values, convert to integer for sorting, do the sort, and get the top three.
highest_latency = (
    log_file
        .map(lambda line: int(re.search(r"latency_ms=([0-9]+)", line).group(1)))
        .sortBy(lambda x: x, ascending=False)
        .take(3)
)

# Display the results.
highest_latency

[5000, 5000, 5000]

16. (2 PTS) Compute the average latency for each service. Ignore log entries without a latency_ms.

In [55]:
# Use RegEx like before.
# NOTE: I again took help from MS Copilot to get this RegEx figured out.
average_latency = (
    log_file
        .map(lambda line: (
            re.search(r"\] (\w+):", line).group(1),
            int(re.search(r"latency_ms=([0-9]+)", line).group(1))
        ))
        .groupByKey()
        .mapValues(lambda x: round(sum(x)/len(x)))
)

# Display the results.
average_latency.collect()

[('api', 2504),
 ('gateway', 2492),
 ('metrics', 2485),
 ('auth', 2507),
 ('scheduler', 2484),
 ('spark', 2508),
 ('storage', 2505),
 ('ingest', 2501)]